In [ ]:
"""
Script para Control de Instrumentos de Medición usando PyVISA

Este script demuestra cómo conectarse y controlar instrumentos de medición
electrónicos (como osciloscopios) usando el protocolo VISA a través de Python.
PyVISA es una biblioteca que permite la comunicación con instrumentos que
siguen el estándar VISA (Virtual Instrument Software Architecture).

Requisitos:
- pyvisa: Biblioteca principal para comunicación VISA
- pyvisa-py: Backend puro de Python para VISA (no requiere drivers nativos)
- pyserial: Para comunicación serie (RS-232, USB)

Autor: [Tu nombre]
Fecha: [Fecha actual]
Versión: 1.0
"""

# Instalación de dependencias necesarias
# Estas bibliotecas permiten la comunicación con instrumentos de medición
!pip install pyvisa pyvisa-py pyserial

import pyvisa

def main():
    """
    Función principal que gestiona la comunicación con un instrumento de medición.
    
    El flujo del programa:
    1. Busca instrumentos disponibles en el sistema
    2. Se conecta a un instrumento específico
    3. Verifica la identidad del instrumento
    4. Lee y modifica parámetros del instrumento (escala de tiempo)
    """
    
    # PASO 1: Inicialización del Administrador de Recursos
    print("=== INICIALIZANDO COMUNICACIÓN CON INSTRUMENTOS ===")
    
    # ResourceManager es el punto de entrada para todas las operaciones VISA
    # Gestiona las conexiones y recursos disponibles en el sistema
    rm = pyvisa.ResourceManager()
    
    # PASO 2: Descubrimiento de Instrumentos
    print("\n--- Buscando instrumentos disponibles ---")
    
    # list_resources() devuelve una tupla con todos los instrumentos detectados
    # Cada instrumento tiene un identificador único (resource string)
    instrumentos = rm.list_resources()
    print(f"Instrumentos encontrados: {instrumentos}")
    
    # Ejemplos de identificadores comunes:
    # - 'ASRL4::INSTR' = Puerto serie COM4
    # - 'USB0::0x1AB1::0x04CE::DS1ZA181804004::INSTR' = Instrumento USB
    # - 'TCPIP0::192.168.1.100::inst0::INSTR' = Instrumento por Ethernet
    
    # PASO 3: Identificación del Instrumento
    print("\n--- Verificando identidad del instrumento ---")
    
    # Definimos el recurso específico al que nos queremos conectar
    # ASRL4::INSTR significa: Puerto Serie Asíncrono número 4, tipo Instrumento
    resource_string = 'ASRL4::INSTR'
    
    try:
        # Abrimos la conexión con el instrumento específico
        # Esto establece un canal de comunicación bidireccional
        instrumento = rm.open_resource(resource_string)
        
        # *IDN? es un comando estándar SCPI que solicita identificación
        # SCPI (Standard Commands for Programmable Instruments) es un protocolo estándar
        # query() envía el comando y espera una respuesta del instrumento
        respuesta = instrumento.query('*IDN?')
        print(f"El instrumento responde: {respuesta}")
        
        # IMPORTANTE: Siempre cerrar la conexión para liberar recursos
        instrumento.close()
        
    except Exception as e:
        print(f"Error al identificar instrumento: {e}")
        # Posibles errores: instrumento no conectado, puerto ocupado, timeout
    
    # PASO 4: Lectura de Parámetros del Instrumento
    print("\n--- Leyendo configuración actual ---")
    
    try:
        # Establecemos nueva conexión (buena práctica para operaciones separadas)
        instrumento = rm.open_resource(resource_string)
        
        # :TIMebase:SCALe? es un comando SCPI específico para osciloscopios
        # Solicita el valor actual de la escala de tiempo (tiempo por división)
        escala_tiempo = instrumento.query(':TIMebase:SCALe?')
        print(f"Escala de tiempo actual: {escala_tiempo}")
        
        # La respuesta típica podría ser algo como "1.00E-03" (1 milisegundo)
        instrumento.close()
        
    except Exception as e:
        print(f"Error al leer configuración: {e}")
    
    # PASO 5: Modificación de Parámetros del Instrumento
    print("\n--- Modificando configuración del instrumento ---")
    
    try:
        instrumento = rm.open_resource(resource_string)
        
        # Configuramos nueva escala de tiempo
        nueva_escala_valor = 5E-3  # 5 milisegundos por división
        print(f"Cambiando escala de tiempo a {nueva_escala_valor}s/div...")
        
        # write() envía un comando sin esperar respuesta
        # :TIMebase:SCALe es el comando para establecer la escala de tiempo
        # 5E-3 significa 5×10^-3 = 0.005 segundos = 5 milisegundos
        instrumento.write(':TIMebase:SCALe 5E-3')
        
        # Verificamos que el cambio se aplicó correctamente
        nueva_escala = instrumento.query(':TIMebase:SCALe?')
        print(f"Nueva escala de tiempo: {nueva_escala}")
        
        # Comparación de valores para verificar éxito
        if abs(float(nueva_escala) - nueva_escala_valor) < 1e-6:
            print("✓ Configuración aplicada correctamente")
        else:
            print("⚠ La configuración podría no haberse aplicado como se esperaba")
        
        instrumento.close()
        
    except Exception as e:
        print(f"Error al modificar configuración: {e}")
        # Errores posibles: valor fuera de rango, comando no soportado
    
    # PASO 6: Limpieza final
    print("\n--- Cerrando administrador de recursos ---")
    
    # Cerramos el administrador de recursos para liberar todos los recursos del sistema
    # Esto es especialmente importante en sistemas con recursos limitados
    rm.close()
    print("✓ Comunicación finalizada correctamente")

def explicacion_comandos():
    """
    Función educativa que explica los comandos SCPI utilizados.
    
    SCPI (Standard Commands for Programmable Instruments) es un estándar
    que define comandos comunes para instrumentos programables.
    """
    
    comandos = {
        '*IDN?': {
            'descripción': 'Solicita identificación del instrumento',
            'respuesta_típica': 'RIGOL TECHNOLOGIES,DS1054Z,DS1ZA181804004,00.04.04.SP3',
            'uso': 'Verificar que el instrumento está conectado y respondiendo'
        },
        
        ':TIMebase:SCALe?': {
            'descripción': 'Consulta la escala de tiempo actual (segundos por división)',
            'respuesta_típica': '1.00E-03 (1 milisegundo por división)',
            'uso': 'Conocer la configuración actual del eje temporal'
        },
        
        ':TIMebase:SCALe <valor>': {
            'descripción': 'Establece la escala de tiempo',
            'ejemplo': ':TIMebase:SCALe 5E-3 (5 milisegundos por división)',
            'uso': 'Ajustar la resolución temporal de las mediciones'
        }
    }
    
    print("\n=== REFERENCIA DE COMANDOS SCPI ===")
    for comando, info in comandos.items():
        print(f"\nComando: {comando}")
        for clave, valor in info.items():
            print(f"  {clave.capitalize()}: {valor}")

def consejos_troubleshooting():
    """
    Consejos para resolver problemas comunes en la comunicación con instrumentos.
    """
    
    print("\n=== GUÍA DE SOLUCIÓN DE PROBLEMAS ===")
    
    problemas_comunes = [
        {
            'problema': 'No se encuentran instrumentos',
            'causas': ['Cable USB desconectado', 'Drivers no instalados', 'Puerto ocupado'],
            'soluciones': ['Verificar conexiones físicas', 'Instalar drivers del fabricante', 'Cerrar otros programas que usen el puerto']
        },
        {
            'problema': 'Timeout en la comunicación',
            'causas': ['Instrumento ocupado', 'Configuración de timeout muy baja'],
            'soluciones': ['Aumentar timeout: instrumento.timeout = 5000', 'Verificar que el instrumento no esté en modo remoto']
        },
        {
            'problema': 'Comando no reconocido',
            'causas': ['Sintaxis incorrecta', 'Comando no soportado por el modelo'],
            'soluciones': ['Consultar manual del instrumento', 'Verificar modelo y firmware']
        }
    ]
    
    for idx, item in enumerate(problemas_comunes, 1):
        print(f"\n{idx}. {item['problema']}")
        print(f"   Causas posibles: {', '.join(item['causas'])}")
        print(f"   Soluciones: {', '.join(item['soluciones'])}")

# Punto de entrada del programa
if __name__ == "__main__":
    main()
    explicacion_comandos()
    consejos_troubleshooting()